# 27 · Bổ sung `age` + `gender` vào `dataset_v3`

`26_add_demographics.ipynb` đã thêm hai cột vào `cohort_v2/cohort_study_level.parquet` (nguồn).
`dataset_v3/*.parquet` là **bản xuất phái sinh** do `07_export_dataset.ipynb` sinh ra, nên hiện
đang lệch — chưa có demographics.

Notebook này nối `age`/`gender` vào các export **theo `study_id`**, không dựng lại toàn bộ export
(việc đó cần checkpoint `fusion_v2` và tính lại embedding).

`v2_embedding_with_support.parquet` không xuất hiện trong bất kỳ notebook nào của repo — nó được
tạo ngoài. Nên notebook này làm việc **theo schema thực tế đọc từ đĩa**, không giả định cách nó
được sinh ra: chỉ cần file có cột `study_id` là nối được.

Cùng 4 lớp an toàn như nb26: **backup → ghi local → kiểm chứng ở local → copy đè lên Drive**.
Cột cũ phải bit-for-bit; sai một chỗ là dừng, Drive chưa bị đụng tới.

**CPU, ~3–8 phút** tuỳ kích thước file. Cần RAM: mỗi export ~1–2 GB khi nạp vào pandas.

## C0 · Config + kiểm kê `dataset_v3`

In [ ]:
import os, json, time, gc, hashlib, shutil
import numpy as np, pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = '.'
OUT_DIR = os.path.join(DATA_DIR, 'cohort_v2')
DS_DIR  = os.path.join(DATA_DIR, 'dataset_v3')
SRC     = os.path.join(OUT_DIR, 'cohort_study_level.parquet')     # da co age/gender

TARGETS = ['v2_embedding.parquet', 'v2_embedding_with_support.parquet']   # <<< sua neu can
NEW_COLS = ['age', 'gender']
KEY = 'study_id'

assert os.path.isdir(DS_DIR), f'khong thay {DS_DIR}'
print(f'nguon  : {SRC}')
print(f'dich   : {DS_DIR}\n')
print('noi dung dataset_v3:')
for f in sorted(os.listdir(DS_DIR)):
    p = os.path.join(DS_DIR, f)
    print(f'  {f:<44} {os.path.getsize(p)/1e6:>9.1f} MB')
_miss = [t for t in TARGETS if not os.path.exists(os.path.join(DS_DIR, t))]
assert not _miss, f'khong thay file dich: {_miss}  -> sua TARGETS cho khop danh sach tren'

## C1 · Nạp `age`/`gender` từ cohort

In [ ]:
DEMO = pd.read_parquet(SRC, columns=[KEY] + NEW_COLS)
assert DEMO[KEY].is_unique, f'{KEY} trong cohort khong duy nhat'
print(f'cohort: {len(DEMO):,} study | age co gia tri {DEMO.age.notna().mean():.2%} '
      f'| gender {dict(DEMO.gender.value_counts(dropna=False))}')

def col_sha(s):
    '''Bam noi dung mot cot, chiu duoc ca cot ragged.'''
    try:
        return hashlib.sha256(
            pd.util.hash_pandas_object(s, index=False).values.tobytes()).hexdigest()
    except TypeError:
        h = hashlib.sha256()
        for v in s.to_numpy():
            if v is None: h.update(b'<NA>')
            elif np.ndim(v) == 0: h.update(b'<NA>' if pd.isna(v) else repr(v).encode())
            else:
                a = np.asarray(v); h.update(str(a.dtype).encode()); h.update(a.tobytes())
            h.update(b'|')
        return h.hexdigest()
print('san sang')

## C2 · Nối vào từng export — backup, ghi local, kiểm chứng, rồi mới đè

Xử lý **tuần tự từng file** và giải phóng RAM giữa các file. Cột mới nối vào **cuối** bảng nên
không xê dịch vị trí cột nào đang có.

In [ ]:
LOCAL_TMP = '/content' if os.path.isdir('/content') else DS_DIR
REPORT = []

for tgt in TARGETS:
    P = os.path.join(DS_DIR, tgt)
    print('\n' + '='*66); print(tgt); print('='*66)
    df = pd.read_parquet(P)
    n0, cols0 = len(df), list(df.columns)
    print(f'  {n0:,} dong x {len(cols0)} cot | RAM {df.memory_usage(deep=True).sum()/1e9:.2f} GB')

    have = [c for c in NEW_COLS if c in cols0]
    if have:
        print(f'  !! da co {have} -> bo qua'); REPORT.append((tgt, 'da co', n0, len(cols0)))
        del df; gc.collect(); continue
    assert KEY in cols0, f'{tgt} khong co cot {KEY} -> khong noi duoc'

    # van tay cot cu (bo qua cot rat rong de tiet kiem thoi gian? -> khong, kiem het)
    sha0 = {c: col_sha(df[c]) for c in cols0}

    m = df[[KEY]].merge(DEMO, on=KEY, how='left')
    assert len(m) == n0 and (m[KEY].to_numpy() == df[KEY].to_numpy()).all(), \
        'merge lam doi so dong hoac thu tu'
    cov = m.age.notna().mean()
    print(f'  khop cohort: {cov:.2%}  ({int(m.age.isna().sum()):,} dong khong khop)')
    assert cov > 0.99, f'chi khop {cov:.2%} -> study_id khong cung khong gian, DUNG'

    for c in NEW_COLS: df[c] = m[c].to_numpy()
    assert list(df.columns) == cols0 + NEW_COLS, 'cot moi khong nam dung cuoi'

    # ---- ghi LOCAL truoc, kiem chung o do ----
    L = os.path.join(LOCAL_TMP, f'_{tgt}')
    df.to_parquet(L, compression='snappy', index=False)
    chk = pd.read_parquet(L)
    bad = [c for c in cols0 if col_sha(chk[c]) != sha0[c]]
    bad += [f'thieu {c}' for c in NEW_COLS if c not in chk.columns]
    assert not bad and len(chk) == n0, \
        f'ban LOCAL sai ({bad}, {len(chk)} dong) -> DUNG, file tren Drive CHUA bi dung toi'
    print(f'  ban local OK ({os.path.getsize(L)/1e6:.1f} MB) -> backup + copy len Drive')
    del chk; gc.collect()

    # ---- backup roi copy de ----
    stamp = time.strftime('%Y%m%d_%H%M%S')
    BAK = P.replace('.parquet', f'.backup_{stamp}.parquet')
    shutil.copyfile(P, BAK)
    shutil.copyfile(L, P)
    for _ in range(10):
        time.sleep(1)
        if os.path.exists(P) and os.path.getsize(P) > 0: break
    os.remove(L)
    print(f'  BACKUP -> {os.path.basename(BAK)}')
    print(f'  GHI    -> {tgt}  ({os.path.getsize(P)/1e6:.1f} MB, '
          f'{len(cols0)} -> {len(cols0)+len(NEW_COLS)} cot)')
    REPORT.append((tgt, os.path.basename(BAK), n0, len(cols0)+len(NEW_COLS)))
    del df; gc.collect()

print('\n' + '='*66)
for t, b, n, c in REPORT: print(f'{t:<44} {n:>8,} dong x {c:>4} cot | {b}')

## C3 · Kiểm chứng sau ghi + cập nhật `columns.json`

In [ ]:
for tgt in TARGETS:
    P = os.path.join(DS_DIR, tgt)
    d = None
    for _try in range(5):                      # cache Drive co the lech vai giay
        try:
            d = pd.read_parquet(P, columns=[KEY] + NEW_COLS); break
        except Exception as e:
            print(f'  doc lai {tgt} lan {_try+1} loi ({type(e).__name__}), doi 3s'); time.sleep(3)
    assert d is not None, (f'khong doc duoc {P}. File CO THE van tot — flush_and_unmount roi '
                           f'mount lai va kiem tay TRUOC KHI phuc hoi tu backup.')
    ok = d.age.notna().mean()
    print(f'{tgt:<44} {len(d):>8,} dong | age {ok:.2%} | gender {d.gender.notna().mean():.2%}')

# ---- cap nhat spec neu co ----
SPEC = os.path.join(DS_DIR, 'columns.json')
if os.path.exists(SPEC):
    spec = json.load(open(SPEC))
    spec.setdefault('demographics', dict(
        names=NEW_COLS, added_by='27_add_demographics_dataset_v3.ipynb',
        source='cohort_v2/cohort_study_level.parquet (them boi nb26)',
        note=('age = anchor_age + (year(index_time) - anchor_year); MIMIC-IV gop tuoi >= 89 '
              'thanh anchor_age=91 nen duoi tren cua age la gia tri DA KIEM DUYET. '
              'gender la chuoi M/F — nb20/nb25 factorize thanh 0/1; neu dung mo hinh TUYEN TINH '
              'thi phai one-hot.'),
        position='noi vao CUOI bang, khong xe dich cot nao dang co',
        applies_to=TARGETS))
    json.dump(spec, open(SPEC, 'w'), indent=1)
    print(f'\ncolumns.json: da them muc "demographics"')
else:
    print('\nkhong thay columns.json -> bo qua')

print(f'''
================================================================
CON LECH — CHUA XU LY

  {DS_DIR}/v2_embedding.npy   (va v1_onehot.npy)

Cac file .npy la MANG THUAN, khong co ten cot, va notebook nay KHONG dung toi.
Neu code phia sau doc .npy thay vi .parquet thi no van chua co age/gender.
Muon dong bo ca .npy thi phai sua 07_export_dataset.ipynb (them khoi DEMO vao EHRX)
roi chay lai export — viec do can checkpoint fusion_v2 va tinh lai embedding.

index.parquet cung chua co age/gender (chi co study_id, split, setting, 13 nhan).
================================================================''')